# 03 — Final submission validation

Load a trained fitted pipeline bundle, align predictions to the official sample ID order and enforce the competition's expected two-column CSV.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if Path('scripts').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
from scripts.predict import predict
from scripts.preprocess import read_competition_data
_, test, sample = read_competition_data(ROOT/'data'/'structural-damage-classification.zip')
bundle = ROOT/'artifacts/notebook02_modular'/'model_bundle.joblib'
if not bundle.exists():
    raise FileNotFoundError('Run notebook 02 or python -m scripts.train --data data first')
submission = predict(ROOT/'data'/'structural-damage-classification.zip',bundle,ROOT/'artifacts/notebook02_modular'/'submission.csv',label_style='numeric')
display(submission.head()); print('Rows:',len(submission))

In [ ]:
import pandas as pd
csv = pd.read_csv(ROOT/'artifacts/notebook02_modular'/'submission.csv', dtype={'building_id':str})
assert list(csv.columns)==['building_id','damage_grade']
assert csv.building_id.equals(sample.building_id.astype(str))
assert csv.damage_grade.isin([1,2,3,4,5]).all()
assert not csv.isna().any().any()
print('Final CSV validated:', ROOT/'artifacts/notebook02_modular'/'submission.csv')
print('Grade frequency:',csv.damage_grade.value_counts().sort_index().to_dict())

Submit this file manually to Kaggle. It is not possible to infer its public/private leaderboard score from its CSV alone. If the actual competition validator requires `Grade 1` strings instead of integers, rerun `python -m scripts.predict --label-style grade` and verify against sample_submission.csv.